In [ ]:
import sys
import os
import math
import numpy as np

# Map each hidden state label to a numeric index
state_index = {"s": 0, "E": 1, "5": 2, "I": 3, "e": 4}
index_to_state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

# Transition probability matrix A[i][j] = P(moving from state i to state j)
transition_matrix = np.array([
    [0.0, 1.0, 0.0, 0.0, 0.0],   # from s
    [0.0, 0.9, 0.1, 0.0, 0.0],   # from E
    [0.0, 0.0, 0.0, 1.0, 0.0],   # from 5
    [0.0, 0.0, 0.0, 0.9, 0.1],   # from I
    [0.0, 0.0, 0.0, 0.0, 0.0],   # from e
])

# Nucleotide symbol -> column index mapping
nuc_to_idx = {"A": 0, "C": 1, "G": 2, "T": 3}

# Emission probability matrix B[state][nucleotide]
emission_matrix = np.array([
    [0.00, 0.00, 0.00, 0.00],   # s  (silent)
    [0.25, 0.25, 0.25, 0.25],   # E  (uniform)
    [0.05, 0.00, 0.95, 0.00],   # 5  (GT-AG rule: mostly G)
    [0.40, 0.10, 0.10, 0.40],   # I  (intron-like)
    [0.00, 0.00, 0.00, 0.00],   # e  (silent)
])

observed_seq = "CTTCATGTGAAAGCAGACGTAAGTCA"
print(f"Observed sequence : {observed_seq}  (length={len(observed_seq)})")
print(f"Number of states  : {len(state_index)}")

In [ ]:
def log_prob_of_path(path: str, seq: str) -> float:
    """
    Compute the joint log-probability of a given state path
    and the observed nucleotide sequence.

    Parameters
    ----------
    path : str
        A string of state labels (e.g. 'EEEEEE5III...').
    seq  : str
        The observed nucleotide sequence.

    Returns
    -------
    float
        Total log-probability (log-scale joint probability).
    """
    # Start with the log of the uniform prior over the first nucleotide
    total_log_prob = math.log(0.25)

    for t in range(1, len(path)):
        prev_idx = state_index[path[t - 1]]
        curr_idx = state_index[path[t]]
        obs_idx  = nuc_to_idx[seq[t]]

        total_log_prob += math.log(
            transition_matrix[prev_idx][curr_idx]
            * emission_matrix[curr_idx][obs_idx]
        )

    return total_log_prob

In [ ]:
# Evaluate several hand-crafted candidate paths to get an intuition
# about which splice-site placement looks most probable.

candidates = {
    "k1": "EEEEEE5IIIIIIIIIIIIIIIIIII",
    "k2": "EEEEEEEE5IIIIIIIIIIIIIIIII",
    "k3": "EEEEEEEEEEEE5IIIIIIIIIIIII",
    "k4": "EEEEEEEEEEEEEEE5IIIIIIIIII",
    "k5": "EEEEEEEEEEEEEEEEEE5IIIIIII",
    "k6": "EEEEEEEEEEEEEEEEEEEEEE5III",
    "only_E": "EEEEEEEEEEEEEEEEEEEEEEEEEE",
}

print(f"{'Label':<10} {'State path':<28} {'Log-prob':>12}")
print("-" * 55)
for label, path in candidates.items():
    lp = log_prob_of_path(path, observed_seq) + math.log(0.1)
    print(f"{label:<10} {path:<28} {lp:>12.5f}")

### Viterbi Matrix Layout

We arrange the Viterbi matrix so that **rows correspond to hidden states** and **columns correspond to positions in the observed sequence**.

Each cell `V[state, t]` holds the **log-probability of the single best path** that ends in `state` after observing the first `t+1` nucleotides.

Below is a schematic for the first two nucleotides `C` and `T`:

```
              C                                          T
s  [ V(s, C)=log P(s→s, C)   max over prev {V(prev,C) + log A[prev→s] + log B[s,T]}  ... ]
E  [ V(E, C)=log P(s→E, C)   max over prev {V(prev,C) + log A[prev→E] + log B[E,T]}  ... ]
5  [ V(5, C)=-inf             max over prev {V(prev,C) + log A[prev→5] + log B[5,T]}  ... ]
I  [ V(I, C)=-inf             max over prev {V(prev,C) + log A[prev→I] + log B[I,T]}  ... ]
e  [ V(e, C)=-inf             max over prev {V(prev,C) + log A[prev→e] + log B[e,T]}  ... ]
```

> All calculations are performed in **log-space** to avoid numerical underflow.

In [ ]:
# ── Initialise the two Viterbi matrices ──────────────────────────────────────
# viterbi_scores : stores the best log-probability reaching each (state, pos)
# viterbi_back   : stores which previous state produced that best score

N_STATES = len(state_index)
T_LEN    = len(observed_seq)
NEG_INF  = float("-inf")

viterbi_scores = np.full((N_STATES, T_LEN), NEG_INF)
viterbi_back   = np.zeros((N_STATES, T_LEN), dtype=int)

# ── Fill column 0 (first nucleotide) ────────────────────────────────────────
init_state  = state_index["s"]
first_obs   = nuc_to_idx[observed_seq[0]]

for st in range(N_STATES):
    t_prob = transition_matrix[init_state][st]
    e_prob = emission_matrix[st][first_obs]
    if t_prob > 0 and e_prob > 0:
        viterbi_scores[st, 0] = math.log(0.25) + math.log(t_prob) + math.log(e_prob)
    viterbi_back[st, 0] = init_state

print(f"Viterbi score matrix shape : {viterbi_scores.shape}")
print(f"Backpointer matrix shape   : {viterbi_back.shape}")
print("\nColumn-0 initialisation (first nucleotide):")
for st in range(N_STATES):
    v = viterbi_scores[st, 0]
    print(f"  state {index_to_state[st]!r:3s}: {v:.4f}" if v != NEG_INF else f"  state {index_to_state[st]!r:3s}: -inf")

### Core Viterbi Recurrence

Implement `best_predecessor()` which computes a **single cell** of the Viterbi matrix.

Given the previous column's scores, the current state, and the current observed nucleotide index, it returns:

1. **`best_score`** — the highest `V[prev, t-1] + log A[prev→curr] + log B[curr, obs]` over all valid predecessor states.
2. **`best_prev_state`** — the index of whichever predecessor state achieved that score.

These two values go directly into `viterbi_scores[curr, t]` and `viterbi_back[curr, t]`.

In [ ]:
def best_predecessor(prev_scores: np.ndarray, curr_state: int, obs_idx: int):
    """
    Compute the Viterbi recurrence for one cell.

    Parameters
    ----------
    prev_scores : 1-D array of length N_STATES
        Log-prob scores from the previous time step.
    curr_state  : int
        Index of the state we are filling in.
    obs_idx     : int
        Index of the observed nucleotide at the current position.

    Returns
    -------
    (best_score, best_prev_state) : (float, int)
    """
    emit = emission_matrix[curr_state][obs_idx]
    if emit == 0.0:
        return NEG_INF, 0   # this state cannot emit the observed nucleotide

    log_emit    = math.log(emit)
    best_score  = NEG_INF
    best_prev   = 0

    for prev_st in range(N_STATES):
        if prev_scores[prev_st] == NEG_INF:
            continue  # predecessor was unreachable
        trans = transition_matrix[prev_st][curr_state]
        if trans == 0.0:
            continue  # forbidden transition

        score = prev_scores[prev_st] + math.log(trans) + log_emit
        if score > best_score:
            best_score = score
            best_prev  = prev_st

    return best_score, best_prev

In [ ]:
# ── Fill columns 1 … T_LEN-1 using the recurrence ───────────────────────────
for t in range(1, T_LEN):
    obs_idx = nuc_to_idx[observed_seq[t]]
    for st in range(N_STATES):
        score, prev = best_predecessor(viterbi_scores[:, t - 1], st, obs_idx)
        viterbi_scores[st, t] = score
        viterbi_back[st, t]   = prev

# ── Pretty-print the completed score matrix ──────────────────────────────────
header = "".join(f"{c:>8}" for c in observed_seq)
print(f"{'':6}{header}")

for st in range(N_STATES):
    row_label = index_to_state[st]
    row_vals  = ""
    for t in range(T_LEN):
        v = viterbi_scores[st, t]
        row_vals += f"{'  -inf':>8}" if v == NEG_INF else f"{v:>8.2f}"
    print(f"{row_label:<6}{row_vals}")

In [ ]:
def traceback_path(score_mat: np.ndarray, back_mat: np.ndarray) -> tuple[str, float]:
    """
    Recover the optimal state sequence via backtracking.

    Starting from the state with the highest score in the last column,
    follow the backpointer matrix to reconstruct the full path.

    Parameters
    ----------
    score_mat : ndarray of shape (N_STATES, T_LEN)
    back_mat  : ndarray of shape (N_STATES, T_LEN), dtype int

    Returns
    -------
    (path_str, best_log_prob) : (str, float)
    """
    # 1. Identify the best terminal state
    terminal_state    = int(np.argmax(score_mat[:, -1]))
    best_log_prob     = score_mat[terminal_state, -1]

    # 2. Walk backwards through the backpointer matrix
    reversed_path = [terminal_state]
    current       = terminal_state

    for t in range(T_LEN - 1, 0, -1):
        current = back_mat[current, t]
        reversed_path.append(current)

    reversed_path.reverse()

    # 3. Convert numeric indices back to state labels
    path_str = "".join(index_to_state[idx] for idx in reversed_path)
    return path_str, best_log_prob


optimal_path, best_log_prob = traceback_path(viterbi_scores, viterbi_back)

print(f"Observed sequence   : {observed_seq}")
print(f"Optimal state path  : {optimal_path}")
print()
print(f"Best path log-prob  : {best_log_prob:.4f}")